GET Request to pull Federal Register Documents

In [1]:
import requests
import pandas as pd

In [2]:
url = "https://www.federalregister.gov/api/v1/documents.json"

params = {
    "conditions[term]": '"artificial intelligence"', 
    "conditions[type][]": ["RULE"],
    "conditions[publication_date][gte]": "2017-01-20",
    "per_page": 1000,
    "order": "oldest",
    "fields[]": ["document_number", "title", "type", "abstract",
                 "publication_date", "agencies", "html_url", "raw_text_url"],
}

In [3]:
# Download all rules (we have 183)
response = requests.get(url, params=params)
response.raise_for_status()           # stop with an error if the request failed
data = response.json()

all_results = data["results"]
print("Downloaded:", len(all_results), "| API reports:", data["count"])

# Check: did we get every document? (Fails if results ever exceed one page)
assert len(all_results) == data["count"]

Downloaded: 183 | API reports: 183


In [4]:
# Turn the results into a table
df = pd.DataFrame(all_results)

# "agencies" holds a list per document; keep just the first agency's name
agency_names = []
for agencies in df["agencies"]:
    if agencies:                              # the list is not empty
        agency_names.append(agencies[0]["name"])
    else:
        agency_names.append(None)
df["agency"] = agency_names

In [5]:
# Label each document's administration
# Dates look like "2023-10-30". Text dates in this format compare correctly, so "2020-05-01" < "2021-01-20" is True.

def label_admin(date):
    if date < "2021-01-20":
        return "Trump I"      # Jan 20, 2017 – Jan 19, 2021
    elif date < "2025-01-20":
        return "Biden"        # Jan 20, 2021 – Jan 19, 2025
    else:
        return "Trump II"     # Jan 20, 2025 – today

df["administration"] = df["publication_date"].apply(label_admin)

# Quick look: how many documents per administration?
print(df["administration"].value_counts())

administration
Biden       101
Trump II     62
Trump I      20
Name: count, dtype: int64


In [6]:
# Check and save
assert df["document_number"].is_unique    # no document appears twice

# Drop the list column (CSV can't store lists well); we kept "agency" instead
df = df.drop(columns=["agencies"])

# Save so other notebooks can load it
df.to_csv("fr_ai_rules.csv", index=False)
print("Saved", len(df), "documents")

Saved 183 documents


In [7]:
# Earliest and latest publication dates in our data
print("First document:", df["publication_date"].min())
print("Last document:", df["publication_date"].max())

First document: 2018-11-23
Last document: 2026-10-13


In [8]:
# Sort by date (oldest first) and show the first 5 documents
df.sort_values("publication_date")[["publication_date", "type", "title"]].head(5)

,publication_date,type,title
0,2018-11-23,Rule,Medicare Program; Revisions to Payment Policie...
1,2019-07-08,Rule,Designation of Music Licensing Collective and ...
2,2019-08-16,Rule,Medicare Program; Hospital Inpatient Prospecti...
3,2019-09-13,Rule,Policy on No-Action Letters
4,2019-11-12,Rule,Medicare Program: Changes to Hospital Outpatie...
